In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve
)

from imblearn.over_sampling import SMOTE
import joblib

In [2]:
fraud_df = pd.read_csv("../data/processed/fraud_features.csv")
credit_df = pd.read_csv("../data/processed/creditcard_cleaned.csv")

print("Fraud data shape:", fraud_df.shape)
print("Credit card data shape:", credit_df.shape)

Fraud data shape: (151112, 17)
Credit card data shape: (283726, 31)


In [3]:
credit_df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
credit_df["Class"].value_counts()

Class
0    283253
1       473
Name: count, dtype: int64

In [5]:
X_credit = credit_df.drop("Class", axis=1)
y_credit = credit_df["Class"]

X_credit_train, X_credit_test, y_credit_train, y_credit_test = train_test_split(
    X_credit,
    y_credit,
    test_size=0.2,
    random_state=42,
    stratify=y_credit
)

print("Training class distribution:")
print(y_credit_train.value_counts())

print("\nTesting class distribution:")
print(y_credit_test.value_counts())

Training class distribution:
Class
0    226602
1       378
Name: count, dtype: int64

Testing class distribution:
Class
0    56651
1       95
Name: count, dtype: int64


In [6]:
scaler_credit = StandardScaler()

X_credit_train_scaled = scaler_credit.fit_transform(X_credit_train)
X_credit_test_scaled = scaler_credit.transform(X_credit_test)

In [7]:
smote = SMOTE(random_state=42)

X_credit_train_resampled, y_credit_train_resampled = smote.fit_resample(
    X_credit_train_scaled,
    y_credit_train
)

print("Before SMOTE:")
print(y_credit_train.value_counts())

print("\nAfter SMOTE:")
print(pd.Series(y_credit_train_resampled).value_counts())

Before SMOTE:
Class
0    226602
1       378
Name: count, dtype: int64

After SMOTE:
Class
0    226602
1    226602
Name: count, dtype: int64


In [8]:
log_reg_credit = LogisticRegression(max_iter=1000, random_state=42)

log_reg_credit.fit(X_credit_train_resampled, y_credit_train_resampled)

y_pred_log_credit = log_reg_credit.predict(X_credit_test_scaled)
y_prob_log_credit = log_reg_credit.predict_proba(X_credit_test_scaled)[:, 1]

In [9]:
log_credit_auc_pr = average_precision_score(y_credit_test, y_prob_log_credit)
log_credit_f1 = f1_score(y_credit_test, y_pred_log_credit)
log_credit_cm = confusion_matrix(y_credit_test, y_pred_log_credit)

print("Logistic Regression - Credit Card Dataset")
print("AUC-PR:", log_credit_auc_pr)
print("F1 Score:", log_credit_f1)
print("\nConfusion Matrix:")
print(log_credit_cm)
print("\nClassification Report:")
print(classification_report(y_credit_test, y_pred_log_credit))

Logistic Regression - Credit Card Dataset
AUC-PR: 0.6750409387333339
F1 Score: 0.1

Confusion Matrix:
[[55169  1482]
 [   12    83]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56651
           1       0.05      0.87      0.10        95

    accuracy                           0.97     56746
   macro avg       0.53      0.92      0.54     56746
weighted avg       1.00      0.97      0.99     56746



In [10]:
rf_credit = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_credit.fit(X_credit_train_resampled, y_credit_train_resampled)

y_pred_rf_credit = rf_credit.predict(X_credit_test_scaled)
y_prob_rf_credit = rf_credit.predict_proba(X_credit_test_scaled)[:, 1]

In [11]:
rf_credit_auc_pr = average_precision_score(y_credit_test, y_prob_rf_credit)
rf_credit_f1 = f1_score(y_credit_test, y_pred_rf_credit)
rf_credit_cm = confusion_matrix(y_credit_test, y_pred_rf_credit)

print("Random Forest - Credit Card Dataset")
print("AUC-PR:", rf_credit_auc_pr)
print("F1 Score:", rf_credit_f1)
print("\nConfusion Matrix:")
print(rf_credit_cm)
print("\nClassification Report:")
print(classification_report(y_credit_test, y_pred_rf_credit))

Random Forest - Credit Card Dataset
AUC-PR: 0.7762333342276246
F1 Score: 0.6610169491525424

Confusion Matrix:
[[56588    63]
 [   17    78]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.55      0.82      0.66        95

    accuracy                           1.00     56746
   macro avg       0.78      0.91      0.83     56746
weighted avg       1.00      1.00      1.00     56746



In [12]:
credit_results = pd.DataFrame({
    "Dataset": ["Credit Card", "Credit Card"],
    "Model": ["Logistic Regression", "Random Forest"],
    "AUC-PR": [log_credit_auc_pr, rf_credit_auc_pr],
    "F1 Score": [log_credit_f1, rf_credit_f1]
})

credit_results

,Dataset,Model,AUC-PR,F1 Score
0,Credit Card,Logistic Regression,0.675041,0.100000
1,Credit Card,Random Forest,0.776233,0.661017


In [13]:
joblib.dump(rf_credit, "../models/random_forest_creditcard.pkl")
joblib.dump(scaler_credit, "../models/scaler_creditcard.pkl")

['../models/scaler_creditcard.pkl']

In [14]:
joblib.dump(log_reg_credit, "../models/logistic_regression_creditcard.pkl")
joblib.dump(scaler_credit, "../models/scaler_creditcard.pkl")

['../models/scaler_creditcard.pkl']

In [15]:
credit_results

,Dataset,Model,AUC-PR,F1 Score
0,Credit Card,Logistic Regression,0.675041,0.100000
1,Credit Card,Random Forest,0.776233,0.661017


In [16]:
best_model_row = credit_results.sort_values(
    by=["AUC-PR", "F1 Score"],
    ascending=False
).iloc[0]

print("Best model name:", best_model_row["Model"])
print("Best model AUC-PR:", best_model_row["AUC-PR"])
print("Best model F1 Score:", best_model_row["F1 Score"])

Best model name: Random Forest
Best model AUC-PR: 0.7762333342276246
Best model F1 Score: 0.6610169491525424
